In [1]:
import os
import glob
import gzip
from pathlib import Path

import pandas as pd
import scanpy as sc
import scirpy as ir
import muon as mu

e:\Anaconda\envs\bio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dir = r"E:\Biology\scRNA\Tumor\human\gex+tcr\GSE139555 Pan-cancer\GSE139555_RAW"

# GEX matrices (10x filtered_feature_bc_matrix.h5)
gex_files = sorted(glob.glob(os.path.join(raw_dir, "*matrix.mtx.gz")))

# TCR contig annotations (10x filtered_contig_annotations.csv.gz)
airr_files = sorted(glob.glob(os.path.join(raw_dir, "*filtered_contig_annotations.csv.gz")))

print(f"Found {len(gex_files)} GEX files")
print(f"Found {len(airr_files)} TCR files")

Found 0 GEX files
Found 0 TCR files


In [3]:
gex_adatas = []
gex_samples = []

for fp in gex_files:
    gex_dir = Path(os.path.dirname(fp))
    # Extract base name: filename without .matrix.mtx.gz extension
    # e.g., "GSM4143655_SAM24345862-lt1.matrix.mtx.gz" -> "GSM4143655_SAM24345862-lt1"
    base_name = os.path.basename(fp).replace(".matrix.mtx.gz", "")
    sample = base_name
    
    # Construct file paths
    matrix_file = gex_dir / f"{base_name}.matrix.mtx.gz"
    barcodes_file = gex_dir / f"{base_name}.barcodes.tsv.gz"
    # Try genes.tsv.gz first (legacy format), then features.tsv.gz (v3+ format)
    genes_file = gex_dir / f"{base_name}.genes.tsv.gz"
    if not genes_file.exists():
        genes_file = gex_dir / f"{base_name}.features.tsv.gz"
    
    # Load barcodes
    with gzip.open(barcodes_file, 'rt') as f:
        barcodes = [line.strip() for line in f]
    
    # Load genes/features
    genes_df = pd.read_csv(genes_file, compression='gzip', sep='\t', header=None)
    if genes_df.shape[1] == 2:
        genes_df.columns = ['gene_ids', 'gene_symbols']
    else:
        genes_df.columns = ['gene_ids', 'gene_symbols', 'feature_type']
    
    # Load matrix
    ad = sc.read_mtx(matrix_file).T  # Transpose to have cells as rows
    
    # Set barcodes and gene names
    ad.obs_names = barcodes
    ad.var_names = genes_df['gene_symbols'].values
    ad.var['gene_ids'] = genes_df['gene_ids'].values
    
    ad.var_names_make_unique()
    gex_adatas.append(ad)
    gex_samples.append(sample)

# Concatenate all samples into one AnnData
# index_unique keeps cell barcodes unique across samples

gex = sc.concat(
    gex_adatas,
    join="outer",
    label="sample",
    keys=gex_samples,
    index_unique="-",
)

gex

ValueError: No objects to concatenate

In [ ]:
gex.obs['sample']

AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGAGTTACCCA-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGCAACACCCG-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGCATCTCCCA-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGGTTCGTCTC-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
                                                            ...            
TTTGTCACACAGGCCT-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCACACCCATTC-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCACACGCTTTC-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCATCACGACTA-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCATCCTCATTA-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
Name: sample, Length: 200626, dtype: category
Categories (32, object): ['GSM4143655_SAM24345862-lt1', 'GSM4143656_SAM24345863-ln1', 'GSM4143657_SAM24348188-lt2', 'G

In [ ]:
airr_adatas = []
airr_samples = []

for fp in airr_files:
    tcr_dir = Path(os.path.dirname(fp))
    base_name = os.path.basename(fp).replace(".filtered_contig_annotations.csv.gz", "")
    sample = base_name
    ad = ir.io.read_10x_vdj(fp)
    airr_adatas.append(ad)
    airr_samples.append(sample)

# Concatenate all samples into one AnnData

airr = sc.concat(
    airr_adatas,
    join="outer",
    label="sample",
    keys=airr_samples,
    index_unique="-",
)

airr

e:\Anaconda\envs\bio\Lib\site-packages\airr\schema.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


e:\Anaconda\envs\bio\Lib\site-packages\anndata\utils.py:362: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)


e:\Anaconda\envs\bio\Lib\site-packages\anndata\utils.py:362: ExperimentalFeatureWarning: Outer joins on awkward.Arrays will have different return values in the future. For details, and to offer input, please see:

	https://github.com/scverse/anndata/issues/898
  warnings.warn(msg, category, stacklevel=stacklevel)


AnnData object with n_obs × n_vars = 123134 × 0
    obs: 'sample'
    obsm: 'airr'

In [ ]:
# Remove .filtered_contig_annotations.csv.gz suffix from obs index
# airr.obs_names = airr.obs_names.str.replace('.filtered_contig_annotations.csv.gz', '', regex=False)

In [ ]:
airr.obs['sample']

cell_id
AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGAGGGCTTCC-2-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGAGTGTTTGC-2-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGAGTTACCCA-1-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
AAACCTGAGTTGTCGT-2-GSM4143655_SAM24345862-lt1    GSM4143655_SAM24345862-lt1
                                                            ...            
TTTGGTTCAAGGTTTC-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGGTTCATCAGTCA-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGGTTTCAACACCA-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCACACAGGCCT-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
TTTGTCATCACGACTA-1-GSM4143686_SAM24363038-rb3    GSM4143686_SAM24363038-rb3
Name: sample, Length: 123134, dtype: category
Categories (32, object): ['GSM4143655_SAM24345862-lt1', 'GSM4143656_SAM24345863-ln1', 'GSM4143657_SAM24348188-

In [ ]:
# Report cell counts - keep all gex cells, filter airr to only cells in gex
common_cells = gex.obs_names.intersection(airr.obs_names)
print(f"GEX cells: {len(gex.obs_names)}")
print(f"AIRR cells: {len(airr.obs_names)}")
print(f"Common cells: {len(common_cells)}")

# Filter airr to only include cells that are also in gex (no airr-only cells)
airr = airr[common_cells, :].copy()
print(f"After filtering - AIRR: {airr.n_obs} cells (GEX kept all {gex.n_obs} cells)")

GEX cells: 200626
AIRR cells: 123134
Common cells: 111062
After filtering - AIRR: 111062 cells (GEX kept all 200626 cells)


In [ ]:
# Create MuData with outer join to keep all cells from gex
mdata = mu.MuData({
    "gex": gex,
    "airr": airr,
})
# # Join meta_airr for cells that have airr data
# mdata.obs = mdata.obs.join(meta_airr, how='left')
# mdata.update()
# print(f"MuData created with {mdata.n_obs} cells")

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [ ]:
import sys
sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
import mdata_utils 


In [ ]:
mdata = mdata_utils.inner_cells_per_mdata(mdata)
mdata

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities

MuData object with n_obs × n_vars = 111062 × 30727
  2 modalities
    gex:	111062 x 30727
      obs:	'sample'
    airr:	111062 x 0
      obs:	'sample'
      obsm:	'airr'

In [ ]:
# Load T cell metadata and merge with mdata
print("Loading T cell metadata...")
tcell_metadata = pd.read_csv("GSE139555_tcell_metadata.txt", sep="\t", index_col=0)
tcell_metadata


Loading T cell metadata...


,UMAP_1,UMAP_2,ident,patient,sample,source,clonotype
LT1_AAACCTGAGGATATAC-1,0.761298,1.301195,8.3a-Trm,Lung1,LT1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,-6.475698,0.690571,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,-1.326519,0.730108,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,-3.510637,1.008227,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,-5.588617,1.632665,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C12
...,...,...,...,...,...,...,...
RB3_TTTGTCACACAGGCCT-1,1.650373,3.015730,8.2-Tem,Renal3,RB3,Blood,renal3.tnb.C38
RB3_TTTGTCACACCCATTC-1,4.788950,-0.804241,8.3b-Trm,Renal3,RB3,Blood,NaN
RB3_TTTGTCACACGCTTTC-1,-1.538078,0.933320,4.4-FOS,Renal3,RB3,Blood,NaN
RB3_TTTGTCATCACGACTA-1,4.730519,-0.530475,8.3b-Trm,Renal3,RB3,Blood,renal3.tnb.C198


In [ ]:
# Keep only 'ident' and 'sample' columns
tcell_metadata = tcell_metadata[['ident', 'patient', 'source']]
# Convert to strings for HDF5 compatibility
for col in tcell_metadata.columns:
    tcell_metadata[col] = tcell_metadata[col].astype(str)
tcell_metadata.head()


C:\Users\a4945\AppData\Local\Temp\ipykernel_14076\2252666671.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tcell_metadata[col] = tcell_metadata[col].astype(str)
C:\Users\a4945\AppData\Local\Temp\ipykernel_14076\2252666671.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tcell_metadata[col] = tcell_metadata[col].astype(str)
C:\Users\a4945\AppData\Local\Temp\ipykernel_14076\2252666671.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

,ident,patient,source
LT1_AAACCTGAGGATATAC-1,8.3a-Trm,Lung1,Tumor
LT1_AAACCTGAGTTACCCA-1,4.3-TCF7,Lung1,Tumor
LT1_AAACCTGCAACACCCG-1,4.4-FOS,Lung1,Tumor
LT1_AAACCTGCATCTCCCA-1,4.4-FOS,Lung1,Tumor
LT1_AAACCTGGTTCGTCTC-1,4.3-TCF7,Lung1,Tumor


In [ ]:
# Transform mdata index to match tcell_metadata format
# From: 'AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1'
# To: 'LT1_AAACCTGAGGATATAC-1'

print("Before transformation:")
print(f"mdata.obs.index sample (first 3): {list(mdata.obs.index[:3])}")

def transform_barcode(barcode):
    """Transform barcode from 'AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1' to 'LT1_AAACCTGAGGATATAC-1'"""
    # Split by '-GSM' to separate barcode and sample info
    if '-GSM' in barcode:
        parts = barcode.split('-GSM', 1)
        barcode_part = parts[0]  # e.g., 'AAACCTGAGGATATAC-1'
        sample_part = 'GSM' + parts[1]  # e.g., 'GSM4143655_SAM24345862-lt1'
        
        # Extract tissue code (last part after last '-', uppercase it)
        tissue_code = sample_part.rsplit('-', 1)[-1].upper()  # e.g., 'lt1' -> 'LT1'
        
        # Combine: tissue_code + '_' + barcode
        return f"{tissue_code}_{barcode_part}"  # e.g., 'LT1_AAACCTGAGGATATAC-1'
    return barcode

# Transform indices for each modality separately (they have different lengths)
new_gex_index = [transform_barcode(bc) for bc in mdata.mod['gex'].obs_names]
new_airr_index = [transform_barcode(bc) for bc in mdata.mod['airr'].obs_names]
new_mdata_index = [transform_barcode(bc) for bc in mdata.obs_names]

# Update obs names for all modalities
mdata.mod['gex'].obs_names = new_gex_index
mdata.mod['airr'].obs_names = new_airr_index
# Update mdata obs index
mdata.obs.index = new_mdata_index

print("\nAfter transformation:")
print(f"mdata.obs.index sample (first 3): {list(mdata.obs.index[:3])}")
print(f"tcell_metadata.index sample (first 3): {list(tcell_metadata.index[:3])}")
print(f"Matching cells: {mdata.obs.index.isin(tcell_metadata.index).sum()}/{len(mdata.obs)}")

Before transformation:
mdata.obs.index sample (first 3): ['AAACCTGAGGATATAC-1-GSM4143655_SAM24345862-lt1', 'AAACCTGAGTTACCCA-1-GSM4143655_SAM24345862-lt1', 'AAACCTGCAACACCCG-1-GSM4143655_SAM24345862-lt1']

After transformation:
mdata.obs.index sample (first 3): ['LT1_AAACCTGAGGATATAC-1', 'LT1_AAACCTGAGTTACCCA-1', 'LT1_AAACCTGCAACACCCG-1']
tcell_metadata.index sample (first 3): ['LT1_AAACCTGAGGATATAC-1', 'LT1_AAACCTGAGTTACCCA-1', 'LT1_AAACCTGCAACACCCG-1']
Matching cells: 103742/111062


In [ ]:
all_metadata = pd.read_csv("GSE139555_all_metadata.txt", sep="\t", index_col=0)
all_metadata

,UMAP_1,UMAP_2,ident,patient,sample,source,clonotype
LT1_AAACCTGAGGATATAC-1,2.102660,-2.834420,7,Lung1,LT1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,5.154273,-1.666514,5,Lung1,LT1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,2.792480,1.432129,12,Lung1,LT1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,3.219543,2.568602,1,Lung1,LT1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,2.544952,4.659250,21,Lung1,LT1,Tumor,lung1.tn.C12
...,...,...,...,...,...,...,...
RB3_TTTGTCACACAGGCCT-1,5.002127,-2.034373,5,Renal3,RB3,Blood,renal3.tnb.C38
RB3_TTTGTCACACCCATTC-1,2.711555,-6.873210,6,Renal3,RB3,Blood,NaN
RB3_TTTGTCACACGCTTTC-1,2.839453,1.408865,1,Renal3,RB3,Blood,NaN
RB3_TTTGTCATCACGACTA-1,1.347771,-5.012864,6,Renal3,RB3,Blood,renal3.tnb.C198


In [ ]:
# Create barcode mapping (handle concatenation suffixes)
mdata_base_to_full = {bc.rsplit('-', 1)[0] if bc.count('-') >= 2 else bc: bc for bc in mdata.obs_names}
mdata_base_to_full = {k: v for k, v in mdata_base_to_full.items() if k not in mdata_base_to_full or mdata_base_to_full[k] == v}


In [ ]:
# Add 'isT' column: True if cell is in tcell_metadata, False otherwise
tcell_set = set(tcell_metadata.index)
mdata.obs['isT'] = mdata.obs.index.isin(tcell_set)

print(f"T cells (isT=True): {mdata.obs['isT'].sum()}")
print(f"Non-T cells (isT=False): {(~mdata.obs['isT']).sum()}")

# Initialize ident column
mdata.obs['ident'] = None

# For T cells: take ident from tcell_metadata
tcell_matched = 0
for meta_bc in tcell_metadata.index:
    mdata_bc = (meta_bc if meta_bc in mdata.obs_names else 
               mdata_base_to_full.get(meta_bc) or 
               (mdata_base_to_full.get(meta_bc.rsplit('-', 1)[0]) if meta_bc.count('-') >= 2 else None))
    if mdata_bc:
        mdata.obs.loc[mdata_bc, 'ident'] = tcell_metadata.loc[meta_bc, 'ident']
        # Also copy patient and source from tcell_metadata
        for col in ['patient', 'source']:
            if col in tcell_metadata.columns:
                if col not in mdata.obs.columns:
                    mdata.obs[col] = None
                mdata.obs.loc[mdata_bc, col] = tcell_metadata.loc[meta_bc, col]
        tcell_matched += 1

# For non-T cells: take ident from all_metadata
all_meta_set = set(all_metadata.index)
nontcell_matched = 0
for idx in mdata.obs.index[~mdata.obs['isT']]:
    # Try to find matching barcode in all_metadata
    meta_bc = (idx if idx in all_meta_set else 
              (idx.rsplit('-', 1)[0] if idx.count('-') >= 1 and idx.rsplit('-', 1)[0] in all_meta_set else None))
    if meta_bc and 'ident' in all_metadata.columns:
        mdata.obs.loc[idx, 'ident'] = str(all_metadata.loc[meta_bc, 'ident'])
        nontcell_matched += 1

print(f"\nT cells matched with tcell_metadata: {tcell_matched}")
print(f"Non-T cells matched with all_metadata: {nontcell_matched}")
print(f"Added columns: isT, ident, patient, source")

T cells (isT=True): 103742
Non-T cells (isT=False): 7320

T cells matched with tcell_metadata: 103742
Non-T cells matched with all_metadata: 7320
Added columns: isT, ident, patient, source


In [ ]:
mdata.obs['ident']

LT1_AAACCTGAGGATATAC-1     8.3a-Trm
LT1_AAACCTGAGTTACCCA-1     4.3-TCF7
LT1_AAACCTGCAACACCCG-1      4.4-FOS
LT1_AAACCTGCATCTCCCA-1      4.4-FOS
LT1_AAACCTGGTTCGTCTC-1     4.3-TCF7
                            ...    
RB3_TTTGGTTCAAGGTTTC-1     8.3c-Trm
RB3_TTTGGTTCATCAGTCA-1     8.1-Teff
RB3_TTTGGTTTCAACACCA-1    4.6a-Treg
RB3_TTTGTCACACAGGCCT-1      8.2-Tem
RB3_TTTGTCATCACGACTA-1     8.3b-Trm
Name: ident, Length: 111062, dtype: object

In [ ]:
import re

def parse_ident(ident_str):
    """Parse ident string to extract type and subtype. Only keep CD4, CD8, others."""
    if pd.isna(ident_str) or ident_str is None or ident_str == 'None':
        return None, None
    
    ident_str = str(ident_str)
    
    # Extract first digit and only keep CD4, CD8, classify others as 'others'
    first_digit_match = re.match(r'^(\d+)', ident_str)
    if first_digit_match:
        first_digit = int(first_digit_match.group(1))
        if first_digit == 4:
            cell_type = 'CD4'
        elif first_digit == 8:
            cell_type = 'CD8'
        else:
            cell_type = 'others'
    else:
        cell_type = 'others'
    
    # Extract subtype (string after '-')
    if '-' in ident_str:
        subtype = ident_str.split('-', 1)[1]
    else:
        subtype = None

    # Exhausted Trm gradations from ident prefix (paper 8.3a/b/c)
    if ident_str.startswith("8.3a"):
        subtype = "Trm_exh_L"
    elif ident_str.startswith("8.3b"):
        subtype = "Trm_exh_M"
    elif ident_str.startswith("8.3c"):
        subtype = "Trm_exh_H"

    return cell_type, subtype

# Apply parsing to all cells
type_list = []
subtype_list = []

for ident_val in mdata.obs['ident']:
    cell_type, subtype = parse_ident(ident_val)
    type_list.append(cell_type)
    subtype_list.append(subtype)


In [ ]:

# Add new columns
mdata.obs['type'] = type_list
mdata.obs['subtype'] = subtype_list

# Convert to categorical for efficiency
mdata.obs['type'] = mdata.obs['type'].astype('category')
mdata.obs['subtype'] = mdata.obs['subtype'].astype('category')


In [ ]:
# Print summary
print(f"Type distribution:")
print(mdata.obs['type'].value_counts())
print(f"\nSubtype distribution (top 10):")
print(mdata.obs['subtype'].value_counts())

Type distribution:
type
CD8       49269
CD4       49182
others    12611
Name: count, dtype: int64

Subtype distribution (top 10):
subtype
TCF7         12973
Tem          12367
FOS          11165
Trm_exh_L    10051
Teff          9064
Trm           8376
Treg          7808
Trm_exh_H     7145
Trm_exh_M     6554
MT            5312
RPL32         4566
IL6ST         4233
Mitosis       1371
KLRB1         1370
Chrom         1347
Name: count, dtype: int64


In [ ]:
mdata.obs['patient'].value_counts()

patient
Lung2     14663
Lung3     12932
Lung6     12846
Lung1     11234
Endo1     10786
Lung4      9431
Renal2     7406
Lung5      5213
Endo2      4212
Endo3      4088
Renal1     4073
Colon1     3190
Renal3     3137
Colon2      531
Name: count, dtype: int64

In [ ]:
mdata.write("GSE139555_Tcells.h5mu")

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
